# Claims Analytics Dashboard

Healthcare Financial Analysis Using Synthetic EHR Data

1. Load data
2. Prepare claims data
3. Financial overview
4. Claims by encounter type
5. Highest-cost patients
6. Highest-cost conditions
7. Cost by organization
8. Cost by provider
9. Cost over time
10. Consultant findings

Answers:

- Total Charges
- Total Payments
- Average Claim
- Highest Cost Patient
- Cost by Payer
- Cost by Provider

## Import

In [184]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

import numpy as np

PROJECT_ROOT = Path(
    r"C:\AI Projects\PopulationHealthWorkbench"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_tables
from src.terminology import SNOMED_MANUAL_MAP

from src.claims_analytics import (
    build_claim_financials,
    calculate_claim_kpis,
    build_claim_kpi_table,
    get_highest_cost_claims,
    calculate_patient_costs,
    calculate_encounter_costs,
    calculate_diagnosis_costs,
    create_clinical_diagnosis_summary,
    create_unmapped_diagnosis_summary,
)

## Load the Data

In [185]:
tables = load_tables()

## Create Working Tables

In [186]:
patients = tables["patients"].copy()
claims = tables["claims"].copy()
claims_transactions = tables["claims_transactions"].copy()
encounters = tables["encounters"].copy()
conditions = tables["conditions"].copy()

## Data Preparation

In [187]:
# ----------------------------------------------------
# Build reusable analytical tables
# ----------------------------------------------------

claim_financials = build_claim_financials(
    claims_transactions
)

claims_metrics = calculate_claim_kpis(
    claims,
    claim_financials
)

claims_kpi_table = build_claim_kpi_table(
    claims_metrics
)

top_claims = get_highest_cost_claims(
    claim_financials,
    claims,
    top_n=10
)

patient_costs = calculate_patient_costs(
    claims_transactions,
    patients
)

encounter_costs = calculate_encounter_costs(
    claim_financials,
    claims_transactions,
    encounters
)

diagnosis_costs, diagnosis_summary = (
    calculate_diagnosis_costs(
        claim_financials,
        claims,
        conditions,
        SNOMED_MANUAL_MAP
    )
)

clinical_diagnosis_summary = (
    create_clinical_diagnosis_summary(
        diagnosis_summary
    )
)

unmapped_summary = (
    create_unmapped_diagnosis_summary(
        diagnosis_summary
    )
)

In [188]:
total_claims = claims_metrics["total_claims"]
claims_with_charges = claims_metrics["claims_with_charges"]
total_charges = claims_metrics["total_charges"]
total_payments = claims_metrics["total_payments"]
average_claim_charge = claims_metrics["average_claim_charge"]
median_claim_charge = claims_metrics["median_claim_charge"]
largest_claim_charge = claims_metrics["largest_claim_charge"]
average_claim_payment = claims_metrics["average_claim_payment"]
largest_claim_payment = claims_metrics["largest_claim_payment"]
payment_to_charge_ratio = claims_metrics["payment_to_charge_ratio"]
unpaid_balance = claims_metrics["unpaid_balance"]

## Data Validation

In [ ]:
print(f"Claims table IDs: {claims['Id'].nunique():,}")
print(f"Transaction claim IDs: {claims_transactions['CLAIMID'].nunique():,}")
print(f"Claims with positive charges: {claims_with_charges:,}")
print(f"Charge total: ${total_charges:,.2f}")
print(f"Payment total: ${total_payments:,.2f}")

Claims table IDs: 2,566
Transaction claim IDs: 2,566
Claims with positive charges: 2,566
Charge total: $2,946,805.81
Payment total: $2,946,805.81


## Executive KPIs

This section summarizes the most important executive metrics describing healthcare utilization, financial activity, and claims volume.

### Executive KPI table

In [189]:
display(claims_kpi_table[["Metric", "Formatted Value"]])

,Metric,Formatted Value
0,Total Claims,"2,566"
1,Claims with Charges,"2,566"
2,Total Charges,"$2,946,805.81"
3,Total Payments,"$2,946,805.81"
4,Average Claim Charge,"$1,148.40"
5,Median Claim Charge,$171.78
6,Largest Claim Charge,"$221,144.95"
7,Average Claim Payment,"$1,148.40"
8,Largest Claim Payment,"$221,144.95"
9,Payment-to-Charge Ratio,100.0%


## HIGH-COST CLAIMS

### Business Question
Which individual claims are responsible for the highest healthcare expenditures, and what opportunities do these high-cost claims present for cost management, payment integrity review, and utilization analysis?

### Executive Visualization

In [191]:
import plotly.express as px

fig = px.bar(
    top_claims,
    x="CLAIMID",
    y="Total_Charge",
    title="Top 10 Highest-Cost Claims",
    text="Total_Charge"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Claim ID",
    yaxis_title="Total Charge (USD)",
    yaxis_tickprefix="$",
    yaxis_tickformat=",.0f",
    margin=dict(t=80)
)

fig.show()

### Dynamic Interpretation


In [192]:

consultant_report = f"""
# Consultant Interpretation

## Executive Assessment

The claims dataset contains **{total_claims:,} claims**, representing **${total_charges:,.0f}** in billed charges and **${total_payments:,.0f}** in recorded payments.

The average claim charge was **${average_claim_charge:,.2f}**, while the largest individual claim reached **${largest_claim_charge:,.2f}**.

Overall, aggregate payment activity represents **{payment_to_charge_ratio:.1%}** of billed charges.

## Key Business Insights

- Claims activity demonstrates the expected concentration of healthcare spending, with a relatively small number of claims contributing disproportionately to total expenditures.
- The largest claim represents **{largest_claim_charge / total_charges:.1%}** of total billed charges.
- Aggregate unpaid balance equals **${unpaid_balance:,.2f}**.
"""

display(Markdown(consultant_report))


# Consultant Interpretation

## Executive Assessment

The claims dataset contains **2,566 claims**, representing **$2,946,806** in billed charges and **$2,946,806** in recorded payments.

The average claim charge was **$1,148.40**, while the largest individual claim reached **$221,144.95**.

Overall, aggregate payment activity represents **100.0%** of billed charges.

## Key Business Insights

- Claims activity demonstrates the expected concentration of healthcare spending, with a relatively small number of claims contributing disproportionately to total expenditures.
- The largest claim represents **7.5%** of total billed charges.
- Aggregate unpaid balance equals **$-0.00**.


In [193]:

# Spending concentration
largest_claim_share = (
    largest_claim_charge / total_charges
    if total_charges > 0
    else np.nan
)

if largest_claim_share >= 0.15:
    concentration_comment = (
        "Healthcare spending is highly concentrated, with the largest claim "
        f"representing {largest_claim_share:.1%} of total billed charges."
    )
elif largest_claim_share >= 0.08:
    concentration_comment = (
        "Healthcare spending shows moderate concentration among higher-cost claims. "
        f"The largest claim represents {largest_claim_share:.1%} of total billed charges."
    )
else:
    concentration_comment = (
        "Healthcare spending appears relatively distributed across claims. "
        f"The largest claim represents {largest_claim_share:.1%} of total billed charges."
    )

# Payment assessment
if payment_to_charge_ratio >= 0.98:
    payment_comment = (
        "Aggregate payment activity appears internally consistent with billed charges."
    )
elif payment_to_charge_ratio >= 0.90:
    payment_comment = (
        "Most billed charges have corresponding payments, although additional "
        "review of outstanding balances may be warranted."
    )
else:
    payment_comment = (
        "Payment activity differs substantially from billed charges, suggesting "
        "potential payment integrity or reimbursement issues."
    )

consultant_report = f"""
## Consultant Interpretation

### Executive Assessment

The dataset contains **{total_claims:,} claims**, representing
**${total_charges:,.2f}** in billed charges and **${total_payments:,.2f}**
in recorded payments.

The average claim charge was **${average_claim_charge:,.2f}**, the median
claim charge was **${median_claim_charge:,.2f}**, and the largest individual
claim was **${largest_claim_charge:,.2f}**.

### Key Business Insights

- {concentration_comment}
- {payment_comment}
- The aggregate unpaid balance was **${unpaid_balance:,.2f}**.
- Claim-level review is still necessary because aggregate balance can conceal
  variation across patients, providers, organizations, and encounter settings.

### Strategic Recommendations

1. Identify the patients associated with the highest-cost claims and assess their
   chronic disease burden, utilization patterns, and care-management needs.
2. Compare spending across inpatient, emergency, ambulatory, urgent-care, and
   wellness encounters to identify major cost drivers.
3. Evaluate provider- and organization-level variation in charges and utilization.
4. Review unpaid, unusually large, or potentially duplicated transactions for
   payment-integrity concerns.
5. Integrate claims and clinical data to support risk stratification and
   value-based care planning.

### Business Value

This analysis demonstrates how transaction-level claims data can be converted
into executive financial metrics and actionable recommendations for cost
management, payment integrity, and population health strategy.

### Important Limitation

This analysis uses synthetic Synthea-generated data for educational and software
development purposes. The workflow reflects real-world analytical methods, but
the numerical findings should not be interpreted as representative of an actual
healthcare population or organization.
"""

display(Markdown(consultant_report))


## Consultant Interpretation

### Executive Assessment

The dataset contains **2,566 claims**, representing
**$2,946,805.81** in billed charges and **$2,946,805.81**
in recorded payments.

The average claim charge was **$1,148.40**, the median
claim charge was **$171.78**, and the largest individual
claim was **$221,144.95**.

### Key Business Insights

- Healthcare spending appears relatively distributed across claims. The largest claim represents 7.5% of total billed charges.
- Aggregate payment activity appears internally consistent with billed charges.
- The aggregate unpaid balance was **$-0.00**.
- Claim-level review is still necessary because aggregate balance can conceal
  variation across patients, providers, organizations, and encounter settings.

### Strategic Recommendations

1. Identify the patients associated with the highest-cost claims and assess their
   chronic disease burden, utilization patterns, and care-management needs.
2. Compare spending across inpatient, emergency, ambulatory, urgent-care, and
   wellness encounters to identify major cost drivers.
3. Evaluate provider- and organization-level variation in charges and utilization.
4. Review unpaid, unusually large, or potentially duplicated transactions for
   payment-integrity concerns.
5. Integrate claims and clinical data to support risk stratification and
   value-based care planning.

### Business Value

This analysis demonstrates how transaction-level claims data can be converted
into executive financial metrics and actionable recommendations for cost
management, payment integrity, and population health strategy.

### Important Limitation

This analysis uses synthetic Synthea-generated data for educational and software
development purposes. The workflow reflects real-world analytical methods, but
the numerical findings should not be interpreted as representative of an actual
healthcare population or organization.


## HIGHEST-COST PATIENTS

### Business Question

Which patients account for the largest share of healthcare spending, and how concentrated are healthcare costs across the population?

Understanding spending concentration helps identify patients who may benefit from care-management, chronic disease programs, and utilization review.

### Top-Patient Table

In [194]:
top_patients = patient_costs.head(10)

top_patients[
    [
        "Patient",
        "Total_Charges",
        "Number_of_Claims",
        "Charge_Transactions",
        "GENDER"
    ]
]

,Patient,Total_Charges,Number_of_Claims,Charge_Transactions,GENDER
0,Patient 001,654932.51,1123,2012,M
1,Patient 002,446026.94,133,313,M
2,Patient 003,335889.95,291,557,M
3,Patient 004,235632.59,46,157,F
4,Patient 005,195565.68,99,319,F
5,Patient 006,193037.28,99,268,M
6,Patient 007,192642.41,177,413,M
7,Patient 008,129114.71,87,216,M
8,Patient 009,122616.88,94,202,M
9,Patient 010,102206.34,51,195,M


### Executive Visualization

In [195]:
fig = px.bar(
    top_patients.sort_values("Total_Charges"),
    x="Total_Charges",
    y="Patient",
    orientation="h",
    title="Top 10 Highest-Cost Patients",
    text="Total_Charges"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Total Charges (USD)",
    yaxis_title="Patient",
    yaxis_tickprefix="$",
    margin=dict(t=80)
)

fig.show()

### Spending concentration

In [196]:
top5_share = (
    top_patients
    .head(5)["Total_Charges"]
    .sum()
    /
    total_charges
)
print(
    f"Top 5 patients account for "
    f"{top5_share:.1%} of total charges."
)

Top 5 patients account for 63.4% of total charges.


### Dynamic interpretation

In [197]:
if top5_share >= 0.50:
    concentration = (
        "Healthcare spending is highly concentrated among a very small group of patients."
    )
elif top5_share >= 0.30:
    concentration = (
        "Healthcare spending demonstrates moderate concentration among higher-cost patients."
    )
else:
    concentration = (
        "Healthcare spending is relatively distributed across the patient population."
    )

In [198]:

display(Markdown(f"""
## Patient Cost Interpretation

{concentration}

The five highest-cost patients account for **{top5_share:.1%}** of all billed healthcare charges.

These patients would be appropriate candidates for detailed clinical review,
care-management programs, and utilization analysis.
"""))


## Patient Cost Interpretation

Healthcare spending is highly concentrated among a very small group of patients.

The five highest-cost patients account for **63.4%** of all billed healthcare charges.

These patients would be appropriate candidates for detailed clinical review,
care-management programs, and utilization analysis.


## COST BY ENCOUNTER TYPE

### Business Question

Which healthcare encounter settings account for the greatest share of healthcare expenditures?

Understanding spending across ambulatory, inpatient, emergency, urgent care, and wellness encounters helps identify major cost drivers and supports resource allocation, payment strategy, and value-based care initiatives.

### Formatted Summary Table

In [199]:
encounter_costs.style.format({
    "Total_Charges": "${:,.2f}",
    "Total_Payments": "${:,.2f}",
    "Average_Charge_per_Claim": "${:,.2f}",
    "Median_Charge_per_Claim": "${:,.2f}",
    "Share_of_Total_Charges": "{:.1%}"
})

,ENCOUNTERCLASS,Total_Charges,Total_Payments,Number_of_Claims,Average_Charge_per_Claim,Median_Charge_per_Claim,Share_of_Total_Charges
0,ambulatory,"$1,609,081.86","$1,609,081.86",1455,"$1,105.90",$245.97,54.6%
1,wellness,"$423,057.95","$423,057.95",501,$844.43,$171.78,14.4%
2,inpatient,"$409,891.16","$409,891.16",85,"$4,822.25",$165.00,13.9%
3,emergency,"$205,445.93","$205,445.93",141,"$1,457.06",$165.00,7.0%
4,outpatient,"$154,283.12","$154,283.12",244,$632.31,$240.18,5.2%
5,hospice,"$66,816.96","$66,816.96",4,"$16,704.24","$14,687.81",2.3%
6,urgentcare,"$64,004.43","$64,004.43",109,$587.20,$240.18,2.2%
7,snf,"$11,024.40","$11,024.40",1,"$11,024.40","$11,024.40",0.4%
8,virtual,"$3,200.00","$3,200.00",26,$123.08,$125.00,0.1%


### Executive Visualization

In [200]:
import plotly.express as px

fig = px.bar(
    encounter_costs,
    x="ENCOUNTERCLASS",
    y="Total_Charges",
    text="Total_Charges",
    title="Healthcare Charges by Encounter Type"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Encounter Type",
    yaxis_title="Total Charges (USD)",
    yaxis_tickprefix="$",
    yaxis_tickformat=",.0f",
    yaxis_range=[
        0,
        encounter_costs["Total_Charges"].max() * 1.15
    ],
    margin=dict(t=90, r=40, b=70, l=90)
)

fig.show()

### Dynamic interpretation

In [201]:

top_encounter = encounter_costs.iloc[0]

top_encounter_class = top_encounter["ENCOUNTERCLASS"]
top_encounter_charges = top_encounter["Total_Charges"]
top_encounter_share = top_encounter["Share_of_Total_Charges"]
top_encounter_claims = int(top_encounter["Number_of_Claims"])

if top_encounter_share >= 0.50:
    encounter_concentration_comment = (
        "Spending is highly concentrated in one encounter setting."
    )
elif top_encounter_share >= 0.30:
    encounter_concentration_comment = (
        "Spending shows moderate concentration in the leading encounter setting."
    )
else:
    encounter_concentration_comment = (
        "Spending is relatively distributed across encounter settings."
    )

display(Markdown(f"""
## Encounter Cost Interpretation

**{top_encounter_class.title()}** encounters generated the highest total charges,
at **${top_encounter_charges:,.2f}**, representing **{top_encounter_share:.1%}**
of all billed charges across **{top_encounter_claims:,} claims**.

{encounter_concentration_comment}

This finding helps identify where utilization-management, payment-strategy, and
care-redesign efforts may have the greatest financial impact. The next analysis
should examine whether these costs reflect appropriate high-acuity care,
chronic disease burden, or potentially preventable utilization.
"""))


## Encounter Cost Interpretation

**Ambulatory** encounters generated the highest total charges,
at **$1,609,081.86**, representing **54.6%**
of all billed charges across **1,455 claims**.

Spending is highly concentrated in one encounter setting.

This finding helps identify where utilization-management, payment-strategy, and
care-redesign efforts may have the greatest financial impact. The next analysis
should examine whether these costs reflect appropriate high-acuity care,
chronic disease burden, or potentially preventable utilization.


## COST BY DIAGNOSIS

### Business Question

Which clinical conditions are associated with the greatest healthcare expenditures?

Linking financial claims with clinical diagnoses helps identify disease-specific cost drivers, prioritize population health initiatives, support risk-adjustment strategies, and target care-management interventions.

In [202]:
import importlib
import src.terminology

importlib.reload(src.terminology)

<module 'src.terminology' from 'C:\\AI Projects\\PopulationHealthWorkbench\\src\\terminology.py'>

In [203]:
from src.terminology import SNOMED_MANUAL_MAP

len(SNOMED_MANUAL_MAP)

15

In [204]:
dir(src.terminology)

['SNOMED_MANUAL_MAP',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__']

### Top Diagnoses

In [205]:
top_diagnoses = clinical_diagnosis_summary.head(10)

In [206]:
top_diagnoses

,DESCRIPTION,Total_Charges,Total_Payments,Number_of_Claims,Unique_Patients,Average_Claim_Cost,Share_of_Total_Charges
0,Chronic kidney disease stage 4 (disorder),398581.76,398581.76,736,2,541.551304,0.135259
1,Gingivitis (disorder),281179.11,281179.11,93,14,3023.431290,0.095418
2,Pulmonary emphysema (disorder),239140.08,239140.08,5,2,47828.016000,0.081152
3,Dependent drug abuse (disorder),139574.70,139574.70,211,7,661.491469,0.047365
4,Impacted molars (disorder),103004.64,103004.64,17,7,6059.096471,0.034955
5,Chronic congestive heart failure (disorder),84206.19,84206.19,167,2,504.228683,0.028575
6,Acute ST segment elevation myocardial infarcti...,56882.59,56882.59,10,2,5688.259000,0.019303
7,Aortic valve regurgitation (disorder),36258.39,36258.39,16,1,2266.149375,0.012304
8,Myocardial infarction (disorder),32649.21,32649.21,8,3,4081.151250,0.011080
9,Polyp of colon (disorder),30886.56,30886.56,2,2,15443.280000,0.010481


In [207]:
unmapped_charge_share = (
    unmapped_summary["Total_Charges"].sum()
    / total_charges
)


### Terminology Data-Quality Note

In [208]:
display(Markdown(f"""
### Terminology Data-Quality Note

A total of **{len(unmapped_summary):,} SNOMED CT concepts** could not be mapped
through the available Synthea conditions table. These concepts represent
**{unmapped_charge_share:.1%}** of total billed charges.

They were retained and labeled explicitly rather than excluded, because removing
them would understate spending and conceal a meaningful terminology-integration
limitation.
"""))


### Terminology Data-Quality Note

A total of **15 SNOMED CT concepts** could not be mapped
through the available Synthea conditions table. These concepts represent
**17.9%** of total billed charges.

They were retained and labeled explicitly rather than excluded, because removing
them would understate spending and conceal a meaningful terminology-integration
limitation.


### Executive visualization

In [209]:
import plotly.express as px

top_diagnoses = clinical_diagnosis_summary.loc[
    ~clinical_diagnosis_summary["DESCRIPTION"]
    .str.startswith("Unmapped SNOMED", na=False)
].head(10)

fig = px.bar(
    top_diagnoses.sort_values("Total_Charges"),
    x="Total_Charges",
    y="DESCRIPTION",
    orientation="h",
    text="Total_Charges",
    title="Top 10 Highest-Cost Clinical Conditions"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Total Charges (USD)",
    yaxis_title="Clinical Condition",
    xaxis_tickprefix="$",
    xaxis_tickformat=",.0f",
    margin=dict(t=90, r=40, b=70, l=220)
)

fig.show()

### Dynamic consultant interpretation

In [210]:

top_dx = top_diagnoses.iloc[0]

display(Markdown(f"""
## Diagnosis Cost Interpretation

The highest-cost primary diagnosis was **{top_dx['DESCRIPTION']}**.

This condition generated **${top_dx['Total_Charges']:,.2f}** in billed charges,
representing **{top_dx['Share_of_Total_Charges']:.1%}** of total healthcare expenditures.

A total of **{int(top_dx['Unique_Patients'])}** unique patients and
**{int(top_dx['Number_of_Claims'])}** claims were associated with this diagnosis.

### Consultant Assessment

High-cost clinical conditions represent important opportunities for:

- Population health management
- Care-management programs
- Risk-adjustment analytics
- Preventive intervention
- Value-based payment strategy

Further analyses should evaluate whether costs are driven by disease severity,
care fragmentation, repeated utilization, or appropriate management of complex patients.
"""))


## Diagnosis Cost Interpretation

The highest-cost primary diagnosis was **Chronic kidney disease stage 4 (disorder)**.

This condition generated **$398,581.76** in billed charges,
representing **13.5%** of total healthcare expenditures.

A total of **2** unique patients and
**736** claims were associated with this diagnosis.

### Consultant Assessment

High-cost clinical conditions represent important opportunities for:

- Population health management
- Care-management programs
- Risk-adjustment analytics
- Preventive intervention
- Value-based payment strategy

Further analyses should evaluate whether costs are driven by disease severity,
care fragmentation, repeated utilization, or appropriate management of complex patients.
